# GR-Prediction TCN — Colab Training

Trains a non-parametric TCN to predict the gain-reduction envelope of the
SSL G-Bus compressor from dry audio input.

- **Dataset**: loaded from **Google Drive** (mounted at `/content/drive`).
- **Model**: deep causal TCN with long receptive field (~2.7 s) to capture
  the slow release dynamics of the SSL compressor.
- **Logging**: TensorBoard + CSV — train/val loss visible live in-notebook.
- **Checkpoints**: saved to Drive so you can pull `last.ckpt` / `best.ckpt`
  straight to your laptop for local testing.

**Runtime**: Select **GPU** (T4 is fine) via *Runtime → Change runtime type*.

---

### Training recipe (aligned with `training_best_practices.md`)

- **Loss** — L1 on the normalised GR envelope + `0.1 · L1` on its first
  difference. The GR-domain loss is the thesis-specific lever (§15 of the
  best-practices doc); L1 is preferred over MSE / smooth-L1 for stability
  on dynamics targets (§1.1).
- **Optimizer** — AdamW @ `lr=1e-3` (§3, §4.1).
- **LR schedule** — `ReduceLROnPlateau(factor=0.5, patience=20)` on
  `loss/val` (§4.2, NablAFx default).
- **Early stopping** — 30-epoch patience on `loss/val`, cap at 200 epochs
  (§4.2 / §12, Simionato & Fasciani 2025).
- **Segment length** — 3 s (> 2.7 s RF so every output sample sees real
  context, §5.2–§5.3).
- **Precision** — fp16 mixed (§10).
- **Grad clipping** — global norm 1.0 (§9).
- **Activations** — PReLU in the TCN trunk, tanh at the output (§7,
  Steinmetz & Reiss 2022).

In [1]:
# ── 0. Install dependencies ──────────────────────────────────────────
# Colab ships torch/torchaudio/numpy/scipy/matplotlib/pandas/tensorboard
# all built against numpy 2.x — we must NOT downgrade numpy or they break
# with "numpy.dtype size changed" ABI errors.
#
# We install only what's needed to import TCN from nablafx:
#   • lightning, torchmetrics            — not preinstalled
#   • soundfile, einops                  — runtime
#   • auraloss, wandb, jsonargparse      — imported by nablafx/core/base_system.py
#                                          (pure python, safe vs numpy 2)
#   • nablafx                            — --no-deps so it doesn't pull
#                                          frechet_audio_distance (→ tensorflow
#                                          → numpy<2, which breaks everything).
#                                          We stub frechet_audio_distance in
#                                          cell 5 instead.
!pip install -q lightning torchmetrics soundfile einops \
               auraloss wandb jsonargparse
!pip install -q --no-deps nablafx

# Belt-and-suspenders: if anything above sneakily downgraded numpy, put it back.
!pip install -q "numpy>=2,<3"

In [2]:
# ── 1. Mount Google Drive & locate dataset ───────────────────────────
#
# The dataset is backed up to Drive from the Macbook Air via
# Google Drive for Desktop, so it shows up in Colab under:
#
#     /content/drive/Computers/Macbook Air/data/Diff-SSL-G-Comp
#
# (NOT under MyDrive — "Computers/<device>" is Drive's backup mount.)
#
# Expected folder layout:
#   Diff-SSL-G-Comp/
#   ├── processed_normalized/*.wav
#   └── processed_ground_truth/<SETTING>/*.wav

import os
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# --- configure if auto-discovery fails ---------------------------------
DRIVE_DATA_ROOT = None      # leave as None → auto-find under /content/drive
SETTING = "threshold_-4_attack_1_release_0.4_ratio_10"
# ------------------------------------------------------------------------

def _find_dataset_root() -> str | None:
    """Locate a folder that contains processed_normalized/ + processed_ground_truth/."""
    import glob as _g
    candidates = _g.glob("/content/drive/*/*/*/Diff-SSL-G-Comp", recursive=False)
    candidates += _g.glob("/content/drive/*/*/Diff-SSL-G-Comp")
    candidates += _g.glob("/content/drive/*/Diff-SSL-G-Comp")
    for c in candidates:
        if (os.path.isdir(os.path.join(c, "processed_normalized"))
                and os.path.isdir(os.path.join(c, "processed_ground_truth"))):
            return c
    return None

if DRIVE_DATA_ROOT is None:
    DRIVE_DATA_ROOT = _find_dataset_root()

DATA_ROOT = DRIVE_DATA_ROOT

if DATA_ROOT is None or not os.path.isdir(DATA_ROOT):
    print("Couldn't auto-find Diff-SSL-G-Comp. Here's what's actually under Drive:\n")
    import subprocess
    for top in ("Computers", "MyDrive", "Shareddrives"):
        p = f"/content/drive/{top}"
        if os.path.isdir(p):
            print(f"--- /content/drive/{top}/ ---")
            print(subprocess.check_output(["ls", p]).decode())
    raise AssertionError(
        "Dataset folder not found.\n"
        "→ Look at the listing above, find the folder that holds "
        "'processed_normalized/' and 'processed_ground_truth/', then hard-set\n"
        "      DRIVE_DATA_ROOT = '/content/drive/.../Diff-SSL-G-Comp'\n"
        "  above and re-run this cell."
    )

print(f"Auto-located dataset at: {DATA_ROOT}")

dry_dir = os.path.join(DATA_ROOT, "processed_normalized")
wet_dir = os.path.join(DATA_ROOT, "processed_ground_truth", SETTING)
assert os.path.isdir(dry_dir), f"Missing dry folder: {dry_dir}"
assert os.path.isdir(wet_dir), f"Missing wet folder: {wet_dir}"

n_dry = len([f for f in os.listdir(dry_dir) if f.endswith(".wav")])
n_wet = len([f for f in os.listdir(wet_dir) if f.endswith(".wav")])
print(f"Dataset root : {DATA_ROOT}")
print(f"Setting      : {SETTING}")
print(f"Dry files    : {n_dry}")
print(f"Wet files    : {n_wet}")

# Persistent output dir — save into the SAME Drive-synced Computers/
# folder the dataset lives in, so checkpoints, plots and logs stream
# straight back to the Macbook Air's local disk via Google Drive for
# Desktop. You can open any *.ckpt on your laptop while Colab keeps
# training.
OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "gr_pred_runs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Auto-located dataset at: /content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp
Dataset root : /content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp
Setting      : threshold_-4_attack_1_release_0.4_ratio_10
Dry files    : 175
Wet files    : 10
Outputs will be saved to: /content/drive/Othercomputers/MacBook Air/data/gr_pred_runs


In [ ]:
# ── 1b. (Optional but recommended) Cache dataset to local SSD ────────
# Copies dry WAVs and pre-computed GR curves (.pt) from Drive to the
# Colab local SSD for ~10–50× faster DataLoader I/O.

import shutil, time
from pathlib import Path
from google.colab import drive as _gdrive

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"
USE_LOCAL_CACHE = True  # set to False to read directly from Drive

def _remount_drive():
    try:
        _gdrive.flush_and_unmount()
    except Exception:
        pass
    _gdrive.mount("/content/drive", force_remount=True)

def _robust_copy(src: Path, dst: Path, max_retries: int = 5):
    """Copy src→dst, tolerating FUSE disconnects by remounting + retrying."""
    for attempt in range(1, max_retries + 1):
        try:
            with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
                shutil.copyfileobj(fsrc, fdst, length=1024 * 1024)
            return
        except OSError as e:
            print(f"  [retry {attempt}/{max_retries}] {src.name}: {e}")
            try: dst.unlink(missing_ok=True)
            except Exception: pass
            time.sleep(2 * attempt)
            if "Transport endpoint" in str(e) or e.errno in (107, 5):
                _remount_drive()
    raise RuntimeError(f"Failed to copy {src} after {max_retries} retries")

def _mirror_pair(src: Path, dst: Path, label: str, i: int, n: int):
    need_copy = not dst.exists() or dst.stat().st_size != src.stat().st_size
    if need_copy:
        _robust_copy(src, dst)
    print(f"  {label}: {i}/{n}  ({src.name}){'  [skip]' if not need_copy else ''}")

if USE_LOCAL_CACHE:
    local_dry = Path(LOCAL_DATA_ROOT) / "processed_normalized"
    local_dry.mkdir(parents=True, exist_ok=True)

    # 1) Discover songs from pre-computed GR curves
    gr_src_dir = Path(DATA_ROOT) / "gr_curves" / SETTING
    assert gr_src_dir.is_dir(), (
        f"No pre-computed GR curves at {gr_src_dir}.\n"
        f"Run locally: python gr_dataset.py --dataset diffssl"
    )
    pt_files = sorted(gr_src_dir.glob("*.pt"))
    songs = [p.stem for p in pt_files]
    print(f"Found {len(songs)} songs with GR curves for '{SETTING}':")
    for s in songs:
        print(f"  • {s}")

    # 2) Cache dry WAVs (only those with matching GR curves)
    print(f"\nCopying {len(songs)} dry files ...")
    for i, song in enumerate(songs, 1):
        dry_src = Path(dry_dir) / f"{song}_UnmasteredWAV.wav"
        dry_dst = local_dry    / f"{song}_UnmasteredWAV.wav"
        if not dry_src.exists():
            print(f"  WARNING: no dry file for '{song}' — skipping")
            continue
        _mirror_pair(dry_src, dry_dst, "dry", i, len(songs))

    # 3) Cache pre-computed GR curves (.pt files, ~33 MB each)
    local_gr = Path(LOCAL_DATA_ROOT) / "gr_curves" / SETTING
    local_gr.mkdir(parents=True, exist_ok=True)
    print(f"\nCopying {len(pt_files)} GR curve files ...")
    for i, pt_src in enumerate(pt_files, 1):
        _mirror_pair(pt_src, local_gr / pt_src.name, "gr", i, len(pt_files))

    DATA_ROOT = LOCAL_DATA_ROOT
    print(f"\nUsing local cache: {DATA_ROOT}")
else:
    print(f"Reading directly from Drive: {DATA_ROOT}")

In [4]:
# ── 2. Upload/import src.dsp_torch into this Colab runtime ──────────

from pathlib import Path
import sys

import torch
import torch.nn.functional as F

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


_SRC_DSP_SOURCE = r'''PARAM_ORDER = ["threshold", "attack", "release", "ratio"]
PARAM_RANGES_LOCAL = {
    "threshold": (-20.0, 0.0),
    "attack": (0.1, 30.0),
    "release": (0.1, 1.6),
    "ratio": (2.0, 10.0),
}
'''

_SRC_DSP_TORCH_SOURCE = r'''"""
PyTorch-based DSP utilities: RMS envelopes, gain reduction, GR normalisation,
and compressor-parameter normalisation.

Intended for training pipelines where the computation graph must stay on GPU.
"""

import torch
import torch.nn.functional as F

from src.dsp import PARAM_ORDER, PARAM_RANGES_LOCAL

# ---------------------------------------------------------------------------
# Gain-reduction normalisation constants
# ---------------------------------------------------------------------------

GR_DB_MIN = -30.0
GR_DB_MAX = 0.0
RMS_WINDOW = 1024


# ---------------------------------------------------------------------------
# RMS helpers
# ---------------------------------------------------------------------------


def windowed_rms(signal: torch.Tensor, window_size: int) -> torch.Tensor:
    """Sample-rate windowed RMS via 1-D convolution.

    Args:
        signal: ``[T]``, ``[C, T]``, or ``[B, C, T]`` audio.
        window_size: RMS analysis window in samples.

    Returns:
        RMS envelope (linear amplitude) with the same rank as ``signal``.
    """
    orig_ndim = signal.ndim
    if orig_ndim == 1:
        sig = signal.view(1, 1, -1)
    elif orig_ndim == 2:
        sig = signal.unsqueeze(0)
    elif orig_ndim == 3:
        sig = signal
    else:
        raise ValueError(f"windowed_rms expects 1/2/3-D input, got {orig_ndim}-D")

    batch, channels, frames = sig.shape
    sq = sig.reshape(batch * channels, 1, frames) ** 2
    kernel = (
        torch.ones(1, 1, window_size, device=sig.device, dtype=sq.dtype)
        / window_size
    )
    pad = window_size - 1
    rms_sq = F.conv1d(sq, kernel, padding=pad)[..., :frames]
    rms = torch.sqrt(rms_sq.clamp(min=1e-10)).reshape(batch, channels, frames)

    if orig_ndim == 1:
        return rms.view(-1)
    if orig_ndim == 2:
        return rms.squeeze(0)
    return rms


# ---------------------------------------------------------------------------
# Gain reduction
# ---------------------------------------------------------------------------


def gain_reduction_db(
    dry: torch.Tensor, wet: torch.Tensor, window_size: int = RMS_WINDOW
) -> torch.Tensor:
    """GR in dB = RMS_dB(wet) − RMS_dB(dry).  Negative means compression.

    Args:
        dry: ``[T]``, ``[C, T]``, or ``[B, C, T]`` dry input audio.
        wet: matching wet compressed output audio.
        window_size: RMS window size.

    Returns:
        Gain-reduction signal in dB with the same rank as the inputs.
    """
    dry_rms = windowed_rms(dry, window_size)
    wet_rms = windowed_rms(wet, window_size)
    dry_db = 20 * torch.log10(dry_rms)
    wet_db = 20 * torch.log10(wet_rms)
    return wet_db - dry_db


# ---------------------------------------------------------------------------
# GR normalisation (for training targets)
# ---------------------------------------------------------------------------


def normalize_gr(gr_db: torch.Tensor) -> torch.Tensor:
    """Map ``[GR_DB_MIN, GR_DB_MAX]`` → ``[-1, 1]``."""
    return (gr_db - GR_DB_MIN) / (GR_DB_MAX - GR_DB_MIN) * 2 - 1


def denormalize_gr(gr_norm: torch.Tensor) -> torch.Tensor:
    """Map ``[-1, 1]`` → ``[GR_DB_MIN, GR_DB_MAX]``."""
    return (gr_norm + 1) / 2 * (GR_DB_MAX - GR_DB_MIN) + GR_DB_MIN


def compute_gr_target_norm(
    dry: torch.Tensor,
    wet: torch.Tensor,
    rms_window: int = RMS_WINDOW,
) -> torch.Tensor:
    """Exact GR target used by training/evaluation: dB GR → normalised clamp."""
    gr_db = gain_reduction_db(dry.float(), wet.float(), rms_window)
    return normalize_gr(gr_db).clamp(-1.0, 1.0)


def gr_norm_to_db(gr_norm: torch.Tensor) -> torch.Tensor:
    """Convert a normalised GR curve back to dB for plotting/evaluation."""
    return denormalize_gr(gr_norm)


# ---------------------------------------------------------------------------
# Compressor-parameter normalisation
# ---------------------------------------------------------------------------


def normalize_params(
    threshold: float,
    attack: float,
    release: float,
    ratio: float,
    ranges: dict | None = None,
) -> torch.Tensor:
    """Map raw compressor parameters to ``[0, 1]`` using *ranges*.

    Default ranges are :data:`PARAM_RANGES_LOCAL`.
    """
    if ranges is None:
        ranges = PARAM_RANGES_LOCAL
    raw = torch.tensor([threshold, attack, release, ratio], dtype=torch.float32)
    for i, key in enumerate(PARAM_ORDER):
        lo, hi = ranges[key]
        raw[i] = (raw[i] - lo) / (hi - lo)
    return raw.clamp(0, 1)
'''

_src_dir = Path("/content/src")
_src_dir.mkdir(parents=True, exist_ok=True)
(_src_dir / "__init__.py").write_text("")
(_src_dir / "dsp.py").write_text(_SRC_DSP_SOURCE)
(_src_dir / "dsp_torch.py").write_text(_SRC_DSP_TORCH_SOURCE)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from src.dsp_torch import (  # noqa: E402
    PARAM_ORDER,
    PARAM_RANGES_LOCAL,
    GR_DB_MIN,
    GR_DB_MAX,
    RMS_WINDOW,
    windowed_rms,
    gain_reduction_db,
    normalize_gr,
    denormalize_gr,
    compute_gr_target_norm,
    gr_norm_to_db,
)

print(f"Uploaded src.dsp_torch helpers to {_src_dir / 'dsp_torch.py'}")

NVIDIA H100 80GB HBM3
Uploaded src.dsp_torch helpers to /content/src/dsp_torch.py


In [ ]:
# ── 3. Dataset & DataModule (pre-computed GR curves) ─────────────────
#
# Loads dry audio from WAVs + pre-computed GR envelopes (dB) from .pt
# files exported by gr_dataset.py.  GR curves are cached in RAM at
# setup (~33 MB per song at 44.1 kHz).
#
# Returns (dry [1,T], gr_db [1,T]).  The Lightning system normalises
# GR to [-1, 1] in _step().

import glob
import soundfile as sf
import torchaudio
import lightning as pl
from torch.utils.data import Dataset, DataLoader
from typing import Optional

SAMPLE_RATE = 44100
# 3 s @ 44.1 kHz. Chosen so it EXCEEDS the TCN's receptive field
# (~2.68 s with num_blocks=10, kernel_size=5, dilation_growth=3) — if
# SAMPLE_LENGTH < RF the model's output samples are dominated by zero
# padding rather than real context, wasting capacity. Also comfortably
# spans the SSL compressor's ~1.6 s release tail (best-practices §5.3).
SAMPLE_LENGTH = 132300
SAMPLE_STRIDE = 132300   # non-overlapping chunks (same as original)


class PrecomputedGRDataset(Dataset):
    def __init__(
        self,
        data_root: str,
        settings_folder: str,
        sample_length: int = SAMPLE_LENGTH,
        sample_stride: int = SAMPLE_STRIDE,
        sample_rate: int = SAMPLE_RATE,
        random_crop: bool = False,
        samples: list[dict] | None = None,
        _gr_cache: dict | None = None,
    ):
        self.data_root = data_root
        self.settings_folder = settings_folder
        self.sample_length = sample_length
        self.sample_stride = sample_stride
        self.sample_rate = sample_rate
        self.random_crop = random_crop

        if samples is not None:
            self.samples = samples
            self._gr_cache = _gr_cache or {}
            return

        dry_dir = os.path.join(data_root, "processed_normalized")
        gr_dir = os.path.join(data_root, "gr_curves", settings_folder)
        assert os.path.isdir(gr_dir), f"No pre-computed GR at {gr_dir}"

        dry_lookup: dict[str, str] = {}
        for p in sorted(glob.glob(os.path.join(dry_dir, "*_UnmasteredWAV.wav"))):
            song = os.path.basename(p).replace("_UnmasteredWAV.wav", "")
            dry_lookup[song] = p

        self._gr_cache: dict[str, torch.Tensor] = {}
        for pt_file in sorted(glob.glob(os.path.join(gr_dir, "*.pt"))):
            song = os.path.splitext(os.path.basename(pt_file))[0]
            if song in dry_lookup:
                rec = torch.load(pt_file, weights_only=False)
                self._gr_cache[song] = rec["gr_db"]

        self.samples: list[dict] = []
        for song in sorted(self._gr_cache.keys()):
            n_frames = int(self._gr_cache[song].shape[-1])
            dry_path = dry_lookup[song]
            if sample_length == -1:
                self.samples.append({
                    "song": song, "dry": dry_path,
                    "offset": 0, "frames": n_frames, "n_frames": n_frames,
                })
                continue
            if n_frames < sample_length:
                continue
            max_start = n_frames - sample_length
            for offset in range(0, max_start + 1, sample_stride):
                self.samples.append({
                    "song": song, "dry": dry_path,
                    "offset": offset, "frames": sample_length,
                    "n_frames": n_frames,
                })
            if self.samples[-1]["offset"] != max_start:
                self.samples.append({
                    "song": song, "dry": dry_path,
                    "offset": max_start, "frames": sample_length,
                    "n_frames": n_frames,
                })

        cache_mb = sum(t.numel() * 4 for t in self._gr_cache.values()) / 1024 / 1024
        print(
            f"PrecomputedGRDataset: {len(self._gr_cache)} songs, "
            f"{len(self.samples)} crops, {cache_mb:.0f} MB cached  "
            f"[setting={settings_folder}]"
        )

    def with_samples(self, samples, random_crop):
        return PrecomputedGRDataset(
            data_root=self.data_root,
            settings_folder=self.settings_folder,
            sample_length=self.sample_length,
            sample_stride=self.sample_stride,
            sample_rate=self.sample_rate,
            random_crop=random_crop,
            samples=samples,
            _gr_cache=self._gr_cache,
        )

    def __len__(self):
        return len(self.samples)

    def _crop_offset(self, sample):
        if self.sample_length == -1 or not self.random_crop:
            return sample["offset"]
        max_start = max(0, sample["n_frames"] - sample["frames"])
        return int(torch.randint(0, max_start + 1, ()).item()) if max_start > 0 else 0

    def __getitem__(self, idx):
        s = self.samples[idx]
        offset = self._crop_offset(s)
        nf = s["frames"] if self.sample_length != -1 else s["n_frames"]

        dry, sr = sf.read(
            s["dry"], start=offset, stop=offset + nf,
            dtype="float32", always_2d=True,
        )
        dry = torch.from_numpy(dry.T)
        if sr != self.sample_rate:
            dry = torchaudio.functional.resample(dry, sr, self.sample_rate)
        if dry.shape[0] > 1:
            dry = dry.mean(dim=0, keepdim=True)

        gr_db = self._gr_cache[s["song"]][..., offset:offset + nf]
        min_len = min(dry.shape[-1], gr_db.shape[-1])
        return dry[..., :min_len], gr_db[..., :min_len]


class GainReductionDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_root: str,
        settings_folder: str,
        sample_length: int = SAMPLE_LENGTH,
        sample_stride: int = SAMPLE_STRIDE,
        sample_rate: int = SAMPLE_RATE,
        train_split: float = 0.8,
        batch_size: int = 16,
        num_workers: int = 2,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.data_root = data_root
        self.settings_folder = settings_folder
        self.sample_length = sample_length
        self.sample_stride = sample_stride
        self.sample_rate = sample_rate
        self.train_split = train_split
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage: Optional[str] = None) -> None:
        full = PrecomputedGRDataset(
            data_root=self.data_root,
            settings_folder=self.settings_folder,
            sample_length=self.sample_length,
            sample_stride=self.sample_stride,
            sample_rate=self.sample_rate,
            random_crop=False,
        )

        songs = sorted({s["song"] for s in full.samples})
        if len(songs) < 2:
            raise ValueError("Need >= 2 songs for train/val split.")
        generator = torch.Generator().manual_seed(42)
        perm = torch.randperm(len(songs), generator=generator).tolist()
        n_train = int(len(songs) * self.train_split)
        n_train = min(max(1, n_train), len(songs) - 1)
        train_songs = {songs[i] for i in perm[:n_train]}
        val_songs = set(songs) - train_songs

        self.train_dataset = full.with_samples(
            [s for s in full.samples if s["song"] in train_songs], random_crop=True)
        self.val_dataset = full.with_samples(
            [s for s in full.samples if s["song"] in val_songs], random_crop=False)
        print(
            f"Train: {len(self.train_dataset)} crops from "
            f"{len(train_songs)} songs {sorted(train_songs)}"
        )
        print(
            f"Val:   {len(self.val_dataset)} crops from "
            f"{len(val_songs)} songs {sorted(val_songs)}"
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers, pin_memory=True, drop_last=True,
            persistent_workers=self.num_workers > 0,
            prefetch_factor=4 if self.num_workers > 0 else None,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True,
            persistent_workers=self.num_workers > 0,
            prefetch_factor=4 if self.num_workers > 0 else None,
        )

In [ ]:
# ── 4. Lightning system ──────────────────────────────────────────────

import types, sys
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

if "rational" not in sys.modules:
    _r = types.ModuleType("rational")
    _rt = types.ModuleType("rational.torch")
    _rt.Rational = type("Rational", (torch.nn.Module,), {"forward": lambda self, x: x})
    _r.torch = _rt
    sys.modules["rational"] = _r
    sys.modules["rational.torch"] = _rt

# frechet_audio_distance pulls in tensorflow (pins numpy<2) → breaks Colab's
# torch/scipy wheels. We never evaluate FAD in this notebook, so stub it.
if "frechet_audio_distance" not in sys.modules:
    _fad = types.ModuleType("frechet_audio_distance")
    _fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
    sys.modules["frechet_audio_distance"] = _fad

from nablafx.processors import TCN


class GRPredictionSystem(pl.LightningModule):
    """
    Loss (best-practices §1.1, §15):
        L = L1(gr_pred, gr_target) + diff_weight · L1(Δgr_pred, Δgr_target)

    Dataset returns (dry, gr_db).  This system normalises gr_db to [-1, 1]
    using normalize_gr() before computing the loss.
    """

    def __init__(
        self,
        processor: torch.nn.Module,
        lr: float = 1e-3,
        diff_weight: float = 0.1,
        lr_patience: int = 20,
        min_lr: float = 1e-5,
    ):
        super().__init__()
        self.processor = processor
        self.lr = lr
        self.diff_weight = diff_weight
        self.lr_patience = lr_patience
        self.min_lr = min_lr

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.processor(x)

    def _step(self, batch: tuple, mode: str) -> torch.Tensor:
        dry, gr_db = batch
        gr_target = normalize_gr(gr_db).clamp(-1.0, 1.0)

        if hasattr(self.processor, "reset_states"):
            self.processor.reset_states()
        gr_pred = self(dry)

        main_loss = F.l1_loss(gr_pred, gr_target)

        dp = gr_pred[..., 1:] - gr_pred[..., :-1]
        dt = gr_target[..., 1:] - gr_target[..., :-1]
        diff_loss = F.l1_loss(dp, dt)

        loss = main_loss + self.diff_weight * diff_loss

        self.log(f"loss/{mode}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"loss/{mode}_main", main_loss, on_step=False, on_epoch=True)
        self.log(f"loss/{mode}_diff", diff_loss, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=self.lr_patience, min_lr=self.min_lr
        )
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "loss/val"}}

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────
#
# Long-context TCN for GR-envelope prediction.
#
# The SSL compressor's release time can reach 1.6 s (~70 000 samples
# @ 44.1 kHz). To "see" the full release tail while remaining causal,
# we need a big receptive field.
#
#   num_blocks=14, kernel_size=5, dilation_growth=2
#   dilations     : 1, 2, 4, …, 2**13 = 8192
#   RF            ≈ 5 + 4·(2+4+…+8192) ≈ 65 533 samples ≈ 1.49 s
#
# That lets a single causal output sample be informed by up to ~1.5 s
# of history — long enough to model the slow release envelope.
#
# Checkpoint strategy (everything lives on Drive under RUN_DIR):
#   • best_cb    → always keeps the top-3 lowest-val-loss snapshots
#   • periodic_cb → dumps `epoch-XXX.ckpt` every few epochs so you can
#                   `rclone`/download any of them MID-TRAINING to test
#                   locally without stopping the Colab job
#   • save_last=True → `last.ckpt` is rewritten every epoch (latest
#                     weights, handy for "how's it doing right now?"
#                     local previews)

import time
from datetime import datetime
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger

# Performance knobs for T4 + long-RF dilated TCN -------------------
torch.backends.cudnn.benchmark = True     # autotune once, then use fastest conv algo
torch.set_float32_matmul_precision("high")

# ╔══════════════════════════════════════════════════════════════════╗
# ║ RESUME?                                                          ║
# ║                                                                  ║
# ║ If Colab disconnected mid-training and you want to pick up       ║
# ║ where you left off, set RESUME_RUN to the folder name of the     ║
# ║ previous run, e.g. "tcn_gr_20260418_104238". Leave as None to    ║
# ║ start fresh.                                                     ║
# ║                                                                  ║
# ║ Resuming restores: weights, AdamW momentum, LR-scheduler state,  ║
# ║ global step / epoch, and best-metric tracking.                   ║
# ╚══════════════════════════════════════════════════════════════════╝
RESUME_RUN: str | None = None
#RESUME_RUN = "tcn_gr_20260426_093002_with_recommendations"

# Short free-form tag appended to the run folder so you can tell runs apart
# at a glance in Drive / `list_runs()`. Keep it filesystem-safe (no spaces).
# Examples: "rf1_5s_n14_d2", "rf2_7s_n10_d3", "no_bn", "bigger_batch".
RUN_TAG: str = "first_gcn"

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR  = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    assert os.path.isfile(_resume_ckpt), (
        f"Can't resume — no last.ckpt in {RUN_DIR}. "
        f"Check the run name or use best-*.ckpt manually."
    )
    print(f"RESUMING run: {RUN_NAME}")
    print(f"  from ckpt : {_resume_ckpt}")
else:
    _ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    RUN_NAME = f"tcn_gr_{_ts}_{RUN_TAG}" if RUN_TAG else f"tcn_gr_{_ts}"
    RUN_DIR  = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
print(f"Run dir (on Drive): {RUN_DIR}")

# ---- hparams -----------------------------------------------------------
# These defaults follow the "safe default" recipe in
# training_best_practices.md (§14). Feel free to tune.
BATCH_SIZE            = 16
LR                    = 1e-3
MAX_EPOCHS            = 1000       # Simionato 2025 cap (best-practices §4.2)
EARLY_STOP_PATIENCE   = 100        # epochs of val-loss patience (§4.2)
LR_PLATEAU_PATIENCE   = 20        # halve LR after this many flat epochs (§4.2)
CKPT_EVERY_N_EPOCHS   = 5         # periodic grab-anytime snapshots

NUM_BLOCKS       = 10
KERNEL_SIZE      = 5
DILATION_GROWTH  = 3
CHANNEL_WIDTH    = 32
ACT_TYPE         = "prelu"        # PReLU in the trunk (best-practices §7)
BATCHNORM        = False          # no FiLM conditioning → keep BN off
BIAS             = True
# ------------------------------------------------------------------------

assert "/drive/" not in DATA_ROOT, (
    f"DATA_ROOT is still pointing at Drive ({DATA_ROOT}). "
    f"Re-run cell 1b (Cache dataset to local SSD) before training, or "
    f"expect ~165 s per batch due to FUSE network reads."
)

dm = GainReductionDataModule(
    data_root=DATA_ROOT,
    settings_folder=SETTING,
    sample_length=SAMPLE_LENGTH,
    sample_stride=SAMPLE_STRIDE,
    sample_rate=SAMPLE_RATE,
    train_split=0.8,
    batch_size=BATCH_SIZE,
    num_workers=2,             # Colab free has 2 CPU cores; 4 thrashes
)

tcn = TCN(
    num_inputs=1,
    num_outputs=1,
    num_controls=0,
    num_blocks=NUM_BLOCKS,
    stack_size=NUM_BLOCKS,       # IMPORTANT: default=10 would wrap dilation
    kernel_size=KERNEL_SIZE,
    dilation_growth=DILATION_GROWTH,
    channel_width=CHANNEL_WIDTH,
    causal=True,
    cond_type=None,
    bias=BIAS,
    batchnorm=BATCHNORM,
    act_type=ACT_TYPE,
)
print(
    f"TCN receptive field: {tcn.rf} samples "
    f"({tcn.rf / SAMPLE_RATE:.3f} s @ {SAMPLE_RATE} Hz)"
)
n_params = sum(p.numel() for p in tcn.parameters())
print(f"TCN parameters    : {n_params:,}")

# Write hparams.json up-front so the monitor notebook can load mid-training
# checkpoints without waiting for trainer.fit to finish.
import hashlib as _hashlib
import json as _json
_src_dsp_torch_sha256 = _hashlib.sha256(_SRC_DSP_TORCH_SOURCE.encode("utf-8")).hexdigest()
_early_hparams = {
    "sample_rate":          SAMPLE_RATE,
    "sample_length":        SAMPLE_LENGTH,
    "rms_window":           RMS_WINDOW,
    "gr_db_min":            GR_DB_MIN,
    "gr_db_max":            GR_DB_MAX,
    "setting":              SETTING,
    "target_function":      "src.dsp_torch.compute_gr_target_norm",
    "src_dsp_torch_file":   "src/dsp_torch.py",
    "src_dsp_torch_sha256": _src_dsp_torch_sha256,
    "batch_size":           BATCH_SIZE,
    "lr":                   LR,
    "max_epochs":           MAX_EPOCHS,
    "early_stop_patience":  EARLY_STOP_PATIENCE,
    "lr_plateau_patience":  LR_PLATEAU_PATIENCE,
    "loss": {
        "kind":        "l1+diff_l1",
        "diff_weight": 0.1,
    },
    "tcn": {
        "num_blocks":      NUM_BLOCKS,
        "stack_size":      NUM_BLOCKS,
        "kernel_size":     KERNEL_SIZE,
        "dilation_growth": DILATION_GROWTH,
        "channel_width":   CHANNEL_WIDTH,
        "causal":          True,
        "batchnorm":       BATCHNORM,
        "cond_type":       None,
        "bias":            BIAS,
        "act_type":        ACT_TYPE,
        "rf_samples":      int(tcn.rf),
        "rf_seconds":      float(tcn.rf / SAMPLE_RATE),
        "num_params":      n_params,
    },
}
with open(os.path.join(RUN_DIR, "hparams.json"), "w") as _f:
    _json.dump(_early_hparams, _f, indent=2)
print(f"Saved hparams → {os.path.join(RUN_DIR, 'hparams.json')}")

system = GRPredictionSystem(
    processor=tcn,
    lr=LR,
    diff_weight=0.1,
    lr_patience=LR_PLATEAU_PATIENCE,
)

# ---- callbacks ---------------------------------------------------------
ckpt_dir = os.path.join(RUN_DIR, "checkpoints")

# A) best-N by validation loss
best_cb = ModelCheckpoint(
    dirpath=ckpt_dir,
    monitor="loss/val",
    mode="min",
    save_top_k=3,
    save_last=True,                     # always also writes last.ckpt
    filename="best-{epoch:03d}-{step}",
    auto_insert_metric_name=False,
)

# B) periodic snapshots — download mid-training without interrupting
periodic_cb = ModelCheckpoint(
    dirpath=ckpt_dir,
    every_n_epochs=CKPT_EVERY_N_EPOCHS,
    save_top_k=-1,                      # keep ALL — disk is Drive, cheap
    filename="epoch-{epoch:03d}",
    auto_insert_metric_name=False,
)

lr_cb = LearningRateMonitor(logging_interval="epoch")

# D) early stopping — cap wall-clock on runs whose val loss has plateaued
#    (best-practices §4.2 / §12, Simionato & Fasciani 2025).
early_stop_cb = EarlyStopping(
    monitor="loss/val",
    mode="min",
    patience=EARLY_STOP_PATIENCE,
    min_delta=0.0,
    verbose=True,
)

# ---- loggers (TensorBoard + CSV, both on Drive) ------------------------
tb_logger  = TensorBoardLogger(save_dir=RUN_DIR, name="tb",  version="")
csv_logger = CSVLogger(       save_dir=RUN_DIR, name="csv", version="")

NEW run: tcn_gr_20260427_094122_with_recommendations_h100
Run dir (on Drive): /content/drive/Othercomputers/MacBook Air/data/gr_pred_runs/tcn_gr_20260427_094122_with_recommendations_h100
TCN receptive field: 118097 samples (2.678 s @ 44100 Hz)
TCN parameters    : 46,913
Saved hparams → /content/drive/Othercomputers/MacBook Air/data/gr_pred_runs/tcn_gr_20260427_094122_with_recommendations_h100/hparams.json


In [8]:
# ---- launch TensorBoard in-notebook ------------------------------------
# %tensorboard magic tokenises on whitespace, so the space in
# "MacBook Air" in RUN_DIR breaks it. Symlink to a space-free path.
_tb_link = "/content/tb_current"
if os.path.islink(_tb_link) or os.path.exists(_tb_link):
    os.remove(_tb_link)
os.symlink(os.path.join(RUN_DIR, "tb"), _tb_link)

%load_ext tensorboard
%tensorboard --logdir /content/tb_current

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices="auto",
    precision="16-mixed",
    callbacks=[best_cb, periodic_cb, lr_cb, early_stop_cb],
    logger=[tb_logger, csv_logger],
    log_every_n_steps=10,
    default_root_dir=RUN_DIR,
    gradient_clip_val=1.0,
)

t0 = time.time()
trainer.fit(system, dm, ckpt_path=_resume_ckpt)   # None → fresh; path → resume
print(f"\nTotal training time: {(time.time() - t0)/60:.1f} min")
print(f"Best val loss: {best_cb.best_model_score:.4f}")
print(f"Best ckpt    : {best_cb.best_model_path}")
print(f"Last ckpt    : {best_cb.last_model_path}")

<IPython.core.display.Javascript object>

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


GainReductionDataset: 10 songs, 877 chunks  [setting=threshold_-4_attack_1_release_0.4_ratio_10]
Train: 701  Val: 176


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ processor │ TCN  │ 46.9 K │ train │     0 │
└───┴───────────┴──────┴────────┴───────┴───────┘

...

INFO: Metric loss/val improved by 0.000 >= min_delta = 0.0. New best score: 0.025
INFO:lightning.pytorch.callbacks.early_stopping:Metric loss/val improved by 0.000 >= min_delta = 0.0. New best score: 0.025


INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 465, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()